# HADIS — PPO Countermeasure Agent Training

**High Altitude Drone Intelligence System**  
Author: Prakash Tiwari | Chandigarh Engineering College (IKGPTU)

This notebook trains a PPO (Proximal Policy Optimization) agent to select
optimal countermeasure actions in the HADIS drone defence scenario.

- **Environment:** `HADISCountermeasureEnv` (7-d state, 5 discrete actions)
- **Algorithm:** PPO with MlpPolicy (Stable-Baselines3)
- **Training:** 1M timesteps with checkpointing and evaluation callbacks

---

In [ ]:
# Cell 2 — Install dependencies
!pip install -q stable-baselines3 gymnasium matplotlib

In [ ]:
# Cell 3 — Mount Google Drive and clone HADIS repo
from google.colab import drive
drive.mount('/content/drive')

import os

REPO_DIR = '/content/HADIS'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Tiwari1782/HADIS.git {REPO_DIR}
    print(f'[HADIS] Repository cloned to {REPO_DIR}')
else:
    print(f'[HADIS] Repository already exists at {REPO_DIR}')

In [ ]:
# Cell 4 — Imports and configuration
import sys
import os
import time

sys.path.append('/content/HADIS')
from config import PATHS, HYPERPARAMS, DRONE_CLASSES, THREAT_LEVELS, NUM_CLASSES

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback
from stable_baselines3.common.monitor import Monitor

# Import HADIS RL environment
sys.path.append(os.path.join('/content/HADIS', 'hadis-ml'))
from rl_agent.env import HADISCountermeasureEnv

print(f'[HADIS] Config loaded')
print(f'[HADIS] RL config: timesteps={HYPERPARAMS["rl_timesteps"]:,}, '
      f'batch={HYPERPARAMS["rl_batch"]}, lr={HYPERPARAMS["rl_learning_rate"]}')

In [ ]:
# Cell 5 — Validate environment with check_env
print('[HADIS] Validating HADISCountermeasureEnv...')

test_env = HADISCountermeasureEnv(isa_csv_path=PATHS['isa_data'])

try:
    check_env(test_env, warn=True, skip_render_check=True)
    print('[HADIS] Environment validation PASSED.')
except Exception as e:
    print(f'[HADIS] Environment validation WARNING: {e}')

# Quick sanity check
obs, info = test_env.reset()
print(f'[HADIS] Observation space: {test_env.observation_space}')
print(f'[HADIS] Action space: {test_env.action_space}')
print(f'[HADIS] Sample obs: {obs}')
print(f'[HADIS] Sample info: {info}')
test_env.close()

In [ ]:
# Cell 6 — Create or resume PPO model
weights_dir = PATHS['weights_rl']
os.makedirs(weights_dir, exist_ok=True)

logs_dir = PATHS['logs']
os.makedirs(logs_dir, exist_ok=True)

# Create monitored environment
train_env = Monitor(
    HADISCountermeasureEnv(isa_csv_path=PATHS['isa_data']),
    filename=os.path.join(logs_dir, 'rl_monitor'),
)

# Check for existing checkpoint
checkpoint_path = os.path.join(weights_dir, 'ppo_hadis_checkpoint.zip')
resume = False

if os.path.exists(checkpoint_path):
    try:
        model = PPO.load(
            checkpoint_path,
            env=train_env,
            device='auto',
        )
        resume = True
        print(f'[HADIS] Resumed PPO model from: {checkpoint_path}')
        print(f'[HADIS] Timesteps so far: {model.num_timesteps:,}')
    except Exception as e:
        print(f'[HADIS] WARNING: Failed to load checkpoint: {e}')
        print('[HADIS] Creating fresh PPO model...')
        resume = False

if not resume:
    model = PPO(
        policy='MlpPolicy',
        env=train_env,
        learning_rate=HYPERPARAMS['rl_learning_rate'],
        batch_size=HYPERPARAMS['rl_batch'],
        n_steps=2048,
        n_epochs=10,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=0.01,
        verbose=1,
        device='auto',
        tensorboard_log=os.path.join(logs_dir, 'rl_tensorboard'),
    )
    print('[HADIS] Fresh PPO model created with MlpPolicy.')

print(f'[HADIS] PPO model ready | Resume: {resume}')

In [ ]:
# Cell 7 — Setup callbacks

# Checkpoint callback: save every 10k steps
checkpoint_callback = CheckpointCallback(
    save_freq=10_000,
    save_path=weights_dir,
    name_prefix='ppo_hadis_checkpoint',
    save_replay_buffer=False,
    save_vecnormalize=False,
)

# Evaluation callback: evaluate every 20k steps
eval_env = Monitor(
    HADISCountermeasureEnv(isa_csv_path=PATHS['isa_data']),
    filename=os.path.join(logs_dir, 'rl_eval_monitor'),
)

eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=weights_dir,
    log_path=logs_dir,
    eval_freq=20_000,
    n_eval_episodes=50,
    deterministic=True,
    render=False,
)

callbacks = [checkpoint_callback, eval_callback]

print('[HADIS] Callbacks configured:')
print(f'  CheckpointCallback: every 10k steps -> {weights_dir}')
print(f'  EvalCallback: every 20k steps, 50 eval episodes')

In [ ]:
# Cell 8 — Train the PPO agent
total_timesteps = HYPERPARAMS['rl_timesteps']

print(f'[HADIS] Starting PPO training for {total_timesteps:,} timesteps...')
print(f'[HADIS] Learning rate: {HYPERPARAMS["rl_learning_rate"]}')
print(f'[HADIS] Batch size: {HYPERPARAMS["rl_batch"]}')
start_time = time.time()

try:
    model.learn(
        total_timesteps=total_timesteps,
        callback=callbacks,
        reset_num_timesteps=False,
        progress_bar=True,
    )
    elapsed = time.time() - start_time
    print(f'[HADIS] Training completed in {elapsed/3600:.1f} hours')
    print(f'[HADIS] Total timesteps: {model.num_timesteps:,}')
except KeyboardInterrupt:
    elapsed = time.time() - start_time
    print(f'[HADIS] Training interrupted after {elapsed/60:.1f} minutes')
    print(f'[HADIS] Timesteps completed: {model.num_timesteps:,}')
except Exception as e:
    elapsed = time.time() - start_time
    print(f'[HADIS] Training error after {elapsed/60:.1f} minutes: {e}')

In [ ]:
# Cell 9 — Save final model
final_path = os.path.join(weights_dir, 'hadis_ppo_final')

try:
    model.save(final_path)
    print(f'[HADIS] Final model saved to: {final_path}.zip')
except Exception as e:
    print(f'[HADIS] WARNING: Failed to save final model: {e}')

# Also save a copy as the checkpoint for future resume
try:
    model.save(os.path.join(weights_dir, 'ppo_hadis_checkpoint'))
    print(f'[HADIS] Checkpoint saved to: {checkpoint_path}')
except Exception as e:
    print(f'[HADIS] WARNING: Failed to save checkpoint: {e}')

In [ ]:
# Cell 10 — Evaluate over 200 episodes
print('[HADIS] Evaluating trained agent over 200 episodes...')

eval_env_final = HADISCountermeasureEnv(isa_csv_path=PATHS['isa_data'])
n_eval_episodes = 200
episode_rewards = []
episode_lengths = []
correct_actions = 0

for ep in range(n_eval_episodes):
    obs, info = eval_env_final.reset(seed=ep)
    total_reward = 0.0
    done = False
    steps = 0
    ep_correct = False

    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = eval_env_final.step(action)
        total_reward += reward
        steps += 1
        done = terminated or truncated

        if info.get('is_correct', False):
            ep_correct = True

    episode_rewards.append(total_reward)
    episode_lengths.append(steps)
    if ep_correct:
        correct_actions += 1

eval_env_final.close()

mean_reward = np.mean(episode_rewards)
std_reward = np.std(episode_rewards)
mean_length = np.mean(episode_lengths)
accuracy = correct_actions / n_eval_episodes

print(f'[HADIS] ========== Evaluation Results (200 episodes) ==========')
print(f'[HADIS] Mean reward:    {mean_reward:.2f} +/- {std_reward:.2f}')
print(f'[HADIS] Mean ep length: {mean_length:.1f} steps')
print(f'[HADIS] Accuracy:       {accuracy:.2%} ({correct_actions}/{n_eval_episodes})')
print(f'[HADIS] =======================================================')
print(f'[HADIS] Final model: {final_path}.zip')

# Plot reward distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(episode_rewards, bins=30, edgecolor='black', alpha=0.7)
axes[0].axvline(mean_reward, color='red', linestyle='--', linewidth=2,
                label=f'Mean: {mean_reward:.1f}')
axes[0].set_xlabel('Episode Reward')
axes[0].set_ylabel('Count')
axes[0].set_title('HADIS PPO - Reward Distribution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].hist(episode_lengths, bins=30, edgecolor='black', alpha=0.7)
axes[1].axvline(mean_length, color='red', linestyle='--', linewidth=2,
                label=f'Mean: {mean_length:.1f}')
axes[1].set_xlabel('Episode Length')
axes[1].set_ylabel('Count')
axes[1].set_title('HADIS PPO - Episode Length Distribution')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()

try:
    plot_path = os.path.join(PATHS['logs'], 'rl_evaluation_results.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    print(f'[HADIS] Evaluation plot saved to: {plot_path}')
except Exception as e:
    print(f'[HADIS] WARNING: Failed to save plot: {e}')

plt.show()